In [5]:
from pathlib import Path
import numpy as np, pandas as pd
from joblib import load, dump

# You are running the notebook from:  xgb/model_results/res_analysis.ipynb
BASE_DIR = Path.cwd()                  # -> xgb/model_results
INPUT_DIR = BASE_DIR / "xgb_res"       # where ACOT2_fixed_percentile_xgboost_LOO_results.joblib etc live

# Outputs will be written next to the notebook
OUTPUT_JOBLIB = BASE_DIR / "full_loo.joblib"
OUTPUT_H5     = BASE_DIR / "full_loo.h5"
OUTPUT_METRICS_CSV = BASE_DIR / "per_protein_metrics.csv"

TOPK_IMPORTANCES = 100                 # keep top-K per LOO row in /importances_topk
DROP_IMPORTANCES_IN_MAIN = True        # drop heavy dicts from /loo table in HDF5

print("INPUT_DIR =", INPUT_DIR)
print("Outputs:", OUTPUT_JOBLIB, OUTPUT_H5, OUTPUT_METRICS_CSV, sep="\n  ")


INPUT_DIR = c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\model_results\xgb_res
Outputs:
  c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\model_results\full_loo.joblib
  c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\model_results\full_loo.h5
  c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\model_results\per_protein_metrics.csv


In [16]:
p = Path("xgb_res/A2M_fixed_percentile_xgboost_LOO_results.joblib")
obj = load(p)                   # usually a pandas DataFrame
type(obj), getattr(obj, "shape", None)

(pandas.core.frame.DataFrame, (369, 7))

In [17]:
obj

,protein,LOO sample,true,predicted,params_dict,features_coef_dict,percentile
0,A2M,MDAMB468_BREAST,-0.345562,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
1,A2M,SH4_SKIN,1.642186,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
2,A2M,AU565_BREAST,-1.129114,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
3,A2M,KMRC1_KIDNEY,1.493333,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
4,A2M,CAL51_BREAST,0.274652,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
...,...,...,...,...,...,...,...
364,A2M,FADU_UPPER_AERODIGESTIVE_TRACT,-1.116991,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
365,A2M,KP4_PANCREAS,1.922976,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
366,A2M,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,-1.106344,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0
367,A2M,OVCAR8_OVARY,-0.770372,NaN,"{'max_depth': 3, 'learning_rate': 0.1, 'n_esti...",None,1.0


In [6]:
files = sorted(INPUT_DIR.glob("*_xgboost_LOO_results.joblib"))
print(f"Found {len(files)} files")
files[:5]


Found 69 files


[WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/model_results/xgb_res/A2M_fixed_percentile_xgboost_LOO_results.joblib'),
 WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/model_results/xgb_res/AAAS_fixed_percentile_xgboost_LOO_results.joblib'),
 WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/model_results/xgb_res/AACS_fixed_percentile_xgboost_LOO_results.joblib'),
 WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/model_results/xgb_res/AAGAB_fixed_percentile_xgboost_LOO_results.joblib'),
 WindowsPath('c:/Users/hp/Desktop/CPPR/Cancer-Protein-RNA-research/xgb/model_results/xgb_res/AAK1_fixed_percentile_xgboost_LOO_results.joblib')]

In [7]:
# Cell 2: helpers
def r2_np(y, yhat):
    y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return np.nan if ss_tot == 0 else 1.0 - ss_res/ss_tot

def try_len_features(imp_obj):
    if isinstance(imp_obj, dict):
        return np.int32(len(imp_obj))
    if isinstance(imp_obj, (list, tuple)):
        return np.int32(len(imp_obj))
    return pd.NA

def flatten_params_col(df, col="params_dict", prefix="param."):
    if col not in df.columns: 
        return df, []
    flat = pd.json_normalize(df[col]).add_prefix(prefix)
    out = pd.concat([df.drop(columns=[col]), flat], axis=1)
    return out, list(flat.columns)

def extract_topk_importances(master_df, k=100):
    rows = []
    if "features_coef_dict" not in master_df.columns or k <= 0:
        return pd.DataFrame(columns=["protein","LOO sample","feature","importance"])
    for _, r in master_df.iterrows():
        obj = r["features_coef_dict"]
        if isinstance(obj, dict) and obj:
            items = sorted(obj.items(), key=lambda kv: kv[1], reverse=True)[:k]
            for feat, val in items:
                rows.append((r["protein"], r["LOO sample"], str(feat), float(val)))
        elif isinstance(obj, (list, tuple)) and obj:
            items = sorted(obj, key=lambda kv: kv[1], reverse=True)[:k]
            for feat, val in items:
                rows.append((r["protein"], r["LOO sample"], str(feat), float(val)))
    return pd.DataFrame(rows, columns=["protein","LOO sample","feature","importance"])

def per_protein_metrics(df):
    gb = df.groupby("protein", sort=False)
    out = []
    for p, g in gb:
        y, yhat = g["true"].to_numpy(float), g["predicted"].to_numpy(float)
        mae = float(np.mean(np.abs(y - yhat)))
        rmse = float(np.sqrt(np.mean((y - yhat)**2)))
        r2 = float(r2_np(y, yhat))
        # correlation can be handy too:
        corr = float(np.corrcoef(y, yhat)[0,1]) if len(g) > 1 else np.nan
        out.append((p, mae, rmse, r2, corr, int(g.shape[0])))
    return pd.DataFrame(out, columns=["protein","MAE","RMSE","R2","PearsonR","N"])


In [11]:
# Cell 3: load & concat
if not files:
    raise SystemExit("No *_xgboost_LOO_results.joblib files found.")

dfs = []
for f in files:
    try:
        df = load(f)
    except Exception as e:
        print(f"[WARN] could not load {f.name}: {e}")
        continue

    if not isinstance(df, pd.DataFrame):
        print(f"[WARN] {f.name} not a DataFrame; skipping.")
        continue

    needed = {"protein","LOO sample","true","predicted","params_dict"}
    if not needed.issubset(df.columns):
        print(f"[WARN] {f.name} missing some of {needed}; skipping.")
        continue

    # flatten params
    df, param_cols = flatten_params_col(df, "params_dict", "param.")

    # derive n_features from importances if present
    if "features_coef_dict" in df.columns:
        df["n_features"] = df["features_coef_dict"].apply(try_len_features).astype("Int32")
    else:
        df["n_features"] = pd.Series([pd.NA]*len(df), dtype="Int32")

    # errors
    df["true"] = df["true"].astype("float32")
    df["predicted"] = df["predicted"].astype("float32")
    df["abs_error"] = (df["true"] - df["predicted"]).abs().astype("float32")
    df["sq_error"] = ((df["true"] - df["predicted"])**2).astype("float32")

    dfs.append(df)

master = pd.concat(dfs, axis=0, ignore_index=True)
print(master.shape)
master


(25461, 21)


,protein,LOO sample,true,predicted,features_coef_dict,percentile,param.max_depth,param.learning_rate,param.n_estimators,param.subsample,...,param.reg_alpha,param.reg_lambda,param.max_bin,param.tree_method,param.predictor,param.feature_selector,param.percentile,n_features,abs_error,sq_error
0,A2M,MDAMB468_BREAST,-0.345562,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
1,A2M,SH4_SKIN,1.642186,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
2,A2M,AU565_BREAST,-1.129114,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
3,A2M,KMRC1_KIDNEY,1.493333,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
4,A2M,CAL51_BREAST,0.274652,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25456,ACSL4,FADU_UPPER_AERODIGESTIVE_TRACT,-0.904069,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
25457,ACSL4,KP4_PANCREAS,-1.120192,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
25458,ACSL4,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.803236,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN
25459,ACSL4,OVCAR8_OVARY,0.322805,NaN,None,1.0,3,0.1,600,0.8,...,0.0,1.0,256,gpu_hist,gpu_predictor,anova,1.0,<NA>,NaN,NaN


In [12]:
# Cell 4: save merged joblib
dump(master, OUTPUT_JOBLIB, compress=3)
print(f"[OK] wrote: {OUTPUT_JOBLIB}  rows={len(master):,}")


[OK] wrote: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\xgb\model_results\full_loo.joblib  rows=25,461


In [13]:
# Cell 5: write HDF5
to_store = master.copy()
if DROP_IMPORTANCES_IN_MAIN and "features_coef_dict" in to_store.columns:
    to_store = to_store.drop(columns=["features_coef_dict"])

# downcast numerics
for c in to_store.columns:
    if to_store[c].dtype == "float64":
        to_store[c] = to_store[c].astype("float32")
    if to_store[c].dtype == "int64":
        to_store[c] = to_store[c].astype("int32")

# write /loo
to_store.to_hdf(OUTPUT_H5, key="loo", mode="w", format="table", complevel=9, complib="blosc")
print(f"[OK] HDF5 /loo -> {OUTPUT_H5}")

# top-K importances
if "features_coef_dict" in master.columns and TOPK_IMPORTANCES > 0:
    imp_topk = extract_topk_importances(master, TOPK_IMPORTANCES)
    if not imp_topk.empty:
        imp_topk["importance"] = imp_topk["importance"].astype("float32")
        imp_topk.to_hdf(OUTPUT_H5, key="importances_topk", format="table", append=True, complevel=9, complib="blosc")
        print(f"[OK] HDF5 /importances_topk rows={len(imp_topk):,}")

# metrics
metrics = per_protein_metrics(master)
metrics.to_hdf(OUTPUT_H5, key="metrics", format="table", append=True, complevel=9, complib="blosc")
metrics.to_csv(OUTPUT_METRICS_CSV, index=False)
print(f"[OK] HDF5 /metrics and CSV -> {OUTPUT_METRICS_CSV}")

# param usage frequencies
param_cols = [c for c in master.columns if c.startswith("param.")]
if param_cols:
    stacks = []
    for c in param_cols:
        vc = master[c].astype(str).value_counts(dropna=False)
        frac = vc / vc.sum()
        stacks.append(pd.DataFrame({"param": c, "value": vc.index, "count": vc.values, "frac": frac.values}))
    param_usage = pd.concat(stacks, axis=0, ignore_index=True)
    param_usage.to_hdf(OUTPUT_H5, key="param_usage", format="table", append=True, complevel=9, complib="blosc")
    print(f"[OK] HDF5 /param_usage rows={len(param_usage):,}")
else:
    print("[INFO] no param.* columns found.")


ValueError: cannot interpret dtype of [Int32]

In [ ]:
# Cell 6: load back
loo = pd.read_hdf(OUTPUT_H5, key="loo")
metrics = pd.read_hdf(OUTPUT_H5, key="metrics")
param_usage = None
try:
    param_usage = pd.read_hdf(OUTPUT_H5, key="param_usage")
except Exception:
    pass

loo.shape, metrics.shape, (param_usage.shape if param_usage is not None else None)


In [ ]:
# Cell 7: accuracy summary
y, yhat = loo["true"].to_numpy(float), loo["predicted"].to_numpy(float)
GLOBAL = {
    "GLOBAL_MAE": float(np.mean(np.abs(y - yhat))),
    "GLOBAL_RMSE": float(np.sqrt(np.mean((y - yhat)**2))),
    "GLOBAL_R2": float(r2_np(y, yhat)),
    "N_rows": int(len(loo)),
    "N_proteins": int(metrics.shape[0]),
}
GLOBAL, metrics.sort_values("RMSE", ascending=False).head(10)


In [ ]:
# Cell 8: selected feature counts
if "n_features" in loo.columns:
    s = loo.loc[loo["n_features"].notna(), "n_features"].astype("float")
    print({
        "median": float(np.nanmedian(s)),
        "mean": float(np.nanmean(s)),
        "p90": float(np.nanpercentile(s, 90)),
        "max": float(np.nanmax(s))
    })


In [ ]:
# Cell 9: param frequencies + lean grid suggestion
if param_usage is not None and not param_usage.empty:
    keys = [k for k in ["param.max_depth","param.colsample_bytree","param.subsample",
                        "param.learning_rate","param.n_estimators",
                        "param.tree_method","param.reg_alpha","param.reg_lambda"]
            if (param_usage["param"]==k).any()]
    # print top 3 per param
    for k in keys:
        sub = param_usage[param_usage["param"]==k].sort_values("frac", ascending=False)
        print(f"\n{k} (top 3):")
        display(sub.head(3))

    # build suggested lean grid: keep dominant value, and 2nd if >= 0.20 share
    reco = {}
    for k in keys:
        sub = param_usage[param_usage["param"]==k].sort_values("frac", ascending=False)
        if sub.empty: 
            continue
        keep = [sub.iloc[0]["value"]]
        if len(sub) > 1 and float(sub.iloc[1]["frac"]) >= 0.20:
            keep.append(sub.iloc[1]["value"])
        reco[k.replace("param.","")] = keep

    print("\nSuggested lean grid dict (paste into your pipeline):")
    print(json.dumps(reco, indent=2))
else:
    print("No param_usage available.")


In [ ]:
# Cell 10: pasteable Python dict for pipeline.py
def python_grid_from_reco(reco):
    # ensure values are properly typed (strings like '0.5' → float)
    def cast(x):
        try:
            return int(x)
        except: 
            try:
                return float(x)
            except:
                return x
    out = {k: [cast(v) for v in vals] for k, vals in reco.items()}
    return out

try:
    lean_grid = python_grid_from_reco(reco)
    print(lean_grid)
except NameError:
    print("Run Cell 9 first to build `reco`.")


In [ ]:
# Cell 11: recommendations based on what you saw
rec = {}

# 1) selected features small? → keep percentile low
if "n_features" in loo.columns:
    nf = loo.loc[loo["n_features"].notna(), "n_features"].astype(float)
    med_nf = float(np.nanmedian(nf)) if len(nf) else np.nan
    if med_nf == med_nf:  # not nan
        if med_nf <= 150:
            rec["percentile"] = "1–2%"
        elif med_nf <= 300:
            rec["percentile"] = "2–3%"
        else:
            rec["percentile"] = "3–5% (dataset-specific)"

# 2) SPLITS: start at 2 unless accuracy falls
rec["SPLITS"] = "2 (bump to 3 only for hard proteins)"

# 3) preselect_k: fewer columns to score p-values on
rec["preselect_k"] = "1000–3000 (start 2000)"

# 4) tree_method: use gpu_hist only if you have a GPU
if "param.tree_method" in loo.columns:
    if (loo["param.tree_method"] == "gpu_hist").mean() > 0.5:
        rec["tree_method"] = "gpu_hist (drop hist)"
    else:
        rec["tree_method"] = "hist (drop gpu_hist if no GPU)"

# 5) n_estimators / early stopping
if "param.n_estimators" in loo.columns:
    ne = loo["param.n_estimators"].astype(float)
    rec["n_estimators"] = f"{int(ne.mode().iloc[0])} (ceiling; early_stopping=50 keeps trees low)"

pd.Series(rec)


In [ ]:
# Cell 12: simple plots (matplotlib only; install if needed)
import matplotlib.pyplot as plt

# true vs predicted (all proteins)
plt.figure()
plt.scatter(loo["true"].astype(float), loo["predicted"].astype(float), s=6, alpha=0.4)
plt.xlabel("True"); plt.ylabel("Predicted"); plt.title("All proteins: True vs Predicted")
plt.show()

# distribution of per-protein R2
plt.figure()
metrics["R2"].hist(bins=30)
plt.xlabel("R2"); plt.ylabel("Proteins"); plt.title("Per-protein R2")
plt.show()

# selected features distribution
if "n_features" in loo.columns:
    plt.figure()
    loo["n_features"].dropna().astype(int).clip(upper=1000).hist(bins=30)
    plt.xlabel("# features used (clipped at 1000)"); plt.ylabel("Rows"); plt.title("Selected features per LOO model")
    plt.show()
